# 12 — Import dependence and refinery-output ratio

Quantify how the composition of supply changed. Report gross imports, net imports, net imports relative to demand, refinery output relative to demand and trade intensity separately.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)

from portugal_refining_resilience.metrics import event_window_summary


In [ ]:
panel = pd.read_csv(PATHS.processed / "fuel_annual_analytical_panel.csv")
metrics_to_plot = ["gross_import_dependence", "net_import_to_demand_ratio", "refinery_output_to_demand_ratio"]
for product in panel["product"].dropna().unique():
    sub = panel.loc[panel["product"] == product].sort_values("year")
    fig, ax = plt.subplots(figsize=(10, 5))
    for metric in metrics_to_plot:
        if metric in sub and sub[metric].notna().any():
            ax.plot(sub["year"], sub[metric], marker="o", label=metric.replace("_", " "))
    ax.axhline(1.0, linewidth=1)
    ax.axvline(2021, linestyle="--", linewidth=1)
    ax.set(title=f"{product}: supply-dependence ratios", xlabel="Year", ylabel="Ratio")
    ax.legend()
    fig.tight_layout()
    fig.savefig(PATHS.figures / f"{product}_dependence_ratios.png", dpi=180)
    plt.show()


In [ ]:
summary_frames = []
for metric in metrics_to_plot:
    if metric in panel.columns:
        summary_frames.append(event_window_summary(panel, value_column=metric, event_year=2021, pre_years=5, post_years=3))
summary = pd.concat(summary_frames, ignore_index=True)
summary["interpretation"] = "descriptive pre/post difference; not causal"
persist_dataframe(summary, PATHS.metrics / "dependence_pre_post_summary.csv")
display(summary)
